# Quality Grader Evaluator - Bugbash Notebook

This notebook tests the **Quality Grader** evaluator (`builtin.quality_grader`) in two modes:
1. **Inline data** – static query/response/context items
2. **Agent responses** – evaluate a live agent's responses

## Prerequisites
```bash
pip install python-dotenv ipykernel azure-cli
pip install "azure-ai-projects>=2.0.0"
az login
```

Set these environment variables (or add them to a `.env` file):
- `FOUNDRY_PROJECT_ENDPOINT` – e.g. `https://<account>.services.ai.azure.com/api/projects/<project>`
- `FOUNDRY_MODEL_NAME` – e.g. `gpt-5-mini`

## Setup & Imports

In [ ]:
import os
import time
from pprint import pprint

from dotenv import load_dotenv
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)
from openai.types.evals.create_eval_completions_run_data_source_param import (
    SourceFileContent as CompletionsSourceFileContent,
    SourceFileContentContent as CompletionsSourceFileContentContent,
)
from openai.types.eval_create_params import DataSourceConfigCustom
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    TestingCriterionAzureAIEvaluator,
    AzureAIDataSourceConfig,
    AzureAIResponsesEvalRunDataSource,
    PromptAgentDefinition,
    ResponseRetrievalItemGenerationParams,
)

load_dotenv()

endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model_deployment_name = os.environ.get("FOUNDRY_MODEL_NAME", "")
print(f"Endpoint: {endpoint}")
print(f"Model: {model_deployment_name}")

In [ ]:
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
client = project_client.get_openai_client()
print("Clients initialized successfully")

## Helper: Wait for eval run to complete and print results

In [ ]:
def wait_and_print_results(client, eval_id, run_id, poll_interval=15):
    """Poll an eval run until completion, then print output items."""
    while True:
        run = client.evals.runs.retrieve(run_id=run_id, eval_id=eval_id)
        if run.status in ("completed", "failed"):
            break
        print(f"  Status: {run.status} – waiting {poll_interval}s...")
        time.sleep(poll_interval)

    print(f"\nEval Run Status: {run.status}")
    print(f"Result Counts: {run.result_counts}")
    print(f"Report URL: {run.report_url}")

    output_items = list(client.evals.runs.output_items.list(run_id=run_id, eval_id=eval_id))
    return run, output_items

---
# Part 1: Inline Data Evaluation

Create an evaluation with a custom data schema and run it against static inline examples.

### 1.1 Create the Evaluation

In [ ]:
inline_data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
            "response": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
            "context": {"type": "string"},
        },
        "required": ["query", "response"],
    },
    include_sample_schema=True,
)

inline_testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="quality_grader",
        evaluator_name="builtin.quality_grader",
        initialization_parameters={"deployment_name": model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.context}}",
        },
    )
]

inline_eval = client.evals.create(
    name="[Bugbash] Quality Grader - Inline Data",
    data_source_config=inline_data_source_config,
    testing_criteria=inline_testing_criteria,  # type: ignore
)
print(f"Evaluation created (id: {inline_eval.id})")

### 1.2 Define test cases

In [ ]:
# --- Success: relevant and complete response (no context) ---
success_query = "What are the benefits of regular exercise?"
success_response = (
    "Regular exercise offers numerous benefits including improved cardiovascular health, "
    "better weight management, enhanced mood through endorphin release, stronger muscles and bones, "
    "and reduced risk of chronic diseases such as diabetes and heart disease."
)

# --- Success: grounded response WITH context ---
success_context = (
    "The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. "
    "It was constructed from 1887 to 1889 as the centerpiece of the 1889 World's Fair. "
    "The tower is 330 metres tall and was the tallest man-made structure in the world until 1930."
)
success_query_ctx = "Tell me about the Eiffel Tower."
success_response_ctx = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris. "
    "It was built between 1887 and 1889 for the 1889 World's Fair and stands 330 metres tall. "
    "It held the record as the tallest man-made structure until 1930."
)

# --- Failure: irrelevant response (no context) ---
failure_query = "How do I reset my password?"
failure_response = "The weather today is sunny with a high of 75 degrees Fahrenheit."

# --- Failure: ungrounded response WITH context ---
failure_context = (
    "Our company's return policy allows returns within 30 days of purchase with a valid receipt. "
    "Items must be in their original packaging and unused condition."
)
failure_query_ctx = "What is the return policy?"
failure_response_ctx = (
    "You can return items within 90 days of purchase without a receipt, "
    "even if the item has been used or opened."
)

# --- Conversation format (list-of-messages) ---
conversation_query = [
    {
        "createdAt": "2025-04-10T10:00:00Z",
        "run_id": "run_qg_bugbash_001",
        "role": "user",
        "content": [{"type": "text", "text": "What is the capital of Japan?"}],
    }
]
conversation_response = [
    {
        "createdAt": "2025-04-10T10:00:05Z",
        "run_id": "run_qg_bugbash_001",
        "role": "assistant",
        "content": [
            {"type": "text", "text": "The capital of Japan is Tokyo. It is the most populous metropolitan area in the world."}
        ],
    }
]

print("Test cases defined")

### 1.3 Create and run the eval run

In [ ]:
inline_run = client.evals.runs.create(
    eval_id=inline_eval.id,
    name="bugbash_inline_data_run",
    metadata={"team": "eval-bugbash", "scenario": "inline-data"},
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(
            type="file_content",
            content=[
                # Success – relevant response, no context
                SourceFileContentContent(item={"query": success_query, "response": success_response, "context": None}),
                # Success – grounded response with context
                SourceFileContentContent(item={"query": success_query_ctx, "response": success_response_ctx, "context": success_context}),
                # Failure – irrelevant response, no context
                SourceFileContentContent(item={"query": failure_query, "response": failure_response, "context": None}),
                # Failure – ungrounded response with context
                SourceFileContentContent(item={"query": failure_query_ctx, "response": failure_response_ctx, "context": failure_context}),
                # Conversation format
                SourceFileContentContent(item={"query": conversation_query, "response": conversation_response, "context": None}),
            ],
        ),
    ),
)
print(f"Eval Run created (id: {inline_run.id})")

In [ ]:
inline_run_result, inline_output_items = wait_and_print_results(client, inline_eval.id, inline_run.id)

---
# Part 2: Agent Response Evaluation

Create an agent, send it a question, then evaluate the agent's response using the Quality Grader.

**Requires** `FOUNDRY_AGENT_NAME` env var (or set it below).

### 2.1 Create or version the agent

In [ ]:
agent_name = "bugbash-qg-agent"

agent_instructions = """You are a customer support assistant for Contoso Electronics.
Answer questions ONLY using the following product information. Do not make up details.

PRODUCT CATALOG:
- Contoso Laptop Pro 15: 15.6" display, Intel i7, 16GB RAM, 512GB SSD, $1,299. Battery life: 12 hours. Weight: 3.5 lbs. 1-year warranty.
- Contoso Wireless Mouse M200: Bluetooth 5.0, ergonomic design, 6-month battery life, $29.99. Compatible with Windows, macOS, and Linux.
- Contoso USB-C Hub 7-in-1: 2x USB-A, 1x USB-C, HDMI 4K@60Hz, SD card reader, ethernet, $49.99. Supports pass-through charging up to 100W.

POLICIES:
- Returns accepted within 30 days with original receipt and packaging.
- Free shipping on orders over $50.
- Warranty covers manufacturing defects only, not accidental damage.
"""

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions=agent_instructions,
    ),
)
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

### 2.2 Send questions to the agent and collect response IDs

In [ ]:
questions = [
    "How much does the Contoso Laptop Pro 15 cost and what's the battery life?",
    "What is the return policy for Contoso Electronics products?",
    "Is the Contoso USB-C Hub compatible with pass-through charging, and what ports does it have?",
]

response_ids = []
for q in questions:
    convo = client.conversations.create(
        items=[{"type": "message", "role": "user", "content": q}],
    )
    resp = client.responses.create(
        conversation=convo.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    response_ids.append(resp.id)
    print(f"Q: {q}")
    print(f"  A: {resp.output_text}")
    print(f"  Response ID: {resp.id}\n")

print(f"Collected {len(response_ids)} response IDs")

### 2.3 Create evaluation for agent responses

In [ ]:
agent_data_source_config = AzureAIDataSourceConfig(type="azure_ai_source", scenario="responses")

agent_testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="quality_grader",
        evaluator_name="builtin.quality_grader",
        initialization_parameters={"deployment_name": model_deployment_name},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    )
]

agent_eval = client.evals.create(
    name="[Bugbash] Quality Grader - Agent Responses",
    data_source_config=agent_data_source_config,
    testing_criteria=agent_testing_criteria,  # type: ignore
)
print(f"Agent Evaluation created (id: {agent_eval.id})")

### 2.4 Run the evaluation against agent responses

In [ ]:
agent_data_source = AzureAIResponsesEvalRunDataSource(
    type="azure_ai_responses",
    item_generation_params=ResponseRetrievalItemGenerationParams(
        type="response_retrieval",
        data_mapping={"response_id": "{{item.resp_id}}"},
        source=CompletionsSourceFileContent(
            type="file_content",
            content=[
                CompletionsSourceFileContentContent(item={"resp_id": rid})
                for rid in response_ids
            ],
        ),
    ),
)

agent_run = client.evals.runs.create(
    eval_id=agent_eval.id,
    name=f"bugbash_agent_run_{agent.name}",
    data_source=agent_data_source,
)
print(f"Agent Eval Run created (id: {agent_run.id})")

In [ ]:
agent_run_result, agent_output_items = wait_and_print_results(client, agent_eval.id, agent_run.id)

---
## Cleanup

Delete the evaluations and agent created during this bugbash.

In [ ]:
# Delete inline eval
client.evals.delete(eval_id=inline_eval.id)
print(f"Inline evaluation deleted (id: {inline_eval.id})")

# Delete agent eval
client.evals.delete(eval_id=agent_eval.id)
print(f"Agent evaluation deleted (id: {agent_eval.id})")

# Delete agent
project_client.agents.delete(agent_name=agent.name)
print(f"Agent deleted (name: {agent.name})")

In [ ]:
# Close clients
client.close()
project_client.close()
credential.close()
print("All clients closed")